# Лабораторная работа 4: Scrapy + Playwright

Два паука: новости со статического сайта и товары с динамического магазина на JavaScript.
Данные проходят через очистку, дедупликацию и валидацию, выгружаются в JSON и CSV.

| Паук | Сайт | Как получает страницу |
|---|---|---|
| `news` | habr.com | обычный запрос Scrapy, страница уже готова (SSR) |
| `ecommerce` | webscraper.io/test-sites/e-commerce | Playwright: ждёт JS и прокручивает страницу |

Установка описана в `README.md`, не забудьте `playwright install chromium`.

**Как получить сохранённый прогон:** `Run All`, дождаться конца, сохранить файл.

In [14]:
import json
import subprocess
import sys
from pathlib import Path

LAB = Path.cwd()
if not (LAB / "run_spider.py").exists() and (LAB / "lab4" / "run_spider.py").exists():
    LAB = LAB / "lab4"        # ядро запустилось из корня проекта

LIMIT = 100

def run(*command, tail=None):
    print("$", " ".join(str(part) for part in command))
    result = subprocess.run(
        [sys.executable, *[str(part) for part in command]],
        cwd=LAB, capture_output=True, text=True, encoding="utf-8", errors="replace",
    )
    lines = (result.stdout + result.stderr).splitlines()
    shown = lines[-tail:] if tail else lines
    print("\n".join(shown))

    # иначе причина падения остаётся выше хвоста и виден только пустой результат
    problems = [line for line in lines
                if ("ERROR" in line or "CRITICAL" in line) and line not in shown]
    if problems:
        print("\nОшибки из полного лога:")
        print("\n".join(problems[:10]))
    return result.returncode

def pipeline_report(spider):
    log = (LAB / "logs" / f"{spider}.log").read_text(encoding="utf-8", errors="replace")
    for line in log.splitlines():
        if "Pipeline:" in line or "ключевым словам" in line or "Карточек" in line:
            print("   ", line.split("] ")[-1])

print("Рабочая папка:", LAB)
print("Интерпретатор:", sys.executable)
print("Python", sys.version.split()[0])
print()

missing = []
for package in ("scrapy", "scrapy_playwright", "playwright", "ftfy", "pytest"):
    try:
        module = __import__(package)
        print(f"{package:18} {getattr(module, '__version__', 'ok')}")
    except ImportError:
        missing.append(package)
        print(f"{package:18} НЕ УСТАНОВЛЕН")

if missing:
    print()
    print("Ядро тетради работает в окружении без этих пакетов:", ", ".join(missing))
    print()
    print("Доустановить прямо в это ядро — отдельной ячейкой:")
    print("    %pip install -r requirements.txt")
    print()
    print("Или собрать окружение лабораторной, в PowerShell из папки lab4:")
    print("    py -m venv .venv")
    print(r"    .\.venv\Scripts\Activate.ps1")
    print("    pip install -r requirements.txt")
    print("затем выбрать ядро .venv кнопкой Select Kernel в правом верхнем углу тетради.")
    raise ModuleNotFoundError("не хватает пакетов: " + ", ".join(missing) + " — инструкция выше")

# pip ставит только обёртку, сам браузер качается отдельной командой
def browser_ready():
    result = subprocess.run(
        [sys.executable, "-m", "playwright", "install", "chromium", "--dry-run"],
        capture_output=True, text=True, encoding="utf-8", errors="replace")
    places = [line.split(":", 1)[1].strip() for line in result.stdout.splitlines()
              if line.strip().startswith("Install location:")]
    return bool(places) and all(Path(place).exists() for place in places)

if browser_ready():
    print("\nБраузер Chromium    скачан")
else:
    print("\nБраузер Chromium    НЕ СКАЧАН")
    print("EcommerceSpider упадёт с Executable doesn't exist и соберёт ноль товаров.")
    print("Выполнить один раз, отдельной ячейкой:")
    print(r"    !{sys.executable} -m playwright install chromium")
    raise RuntimeError("не скачан браузер Chromium — инструкция выше")

Рабочая папка: c:\projects\web-mining\lab4
Интерпретатор: c:\Users\HONOR\AppData\Local\Programs\Python\Python312\python.exe
Python 3.12.7

scrapy             2.19.0
scrapy_playwright  0.0.48
playwright         ok
ftfy               6.3.1
pytest             9.1.1

Браузер Chromium    скачан


---
## Юнит-тесты

Сеть не нужна: HTML подставляется через `HtmlResponse`, pipelines проверяются напрямую.

In [15]:
run("-m", "pytest", "-q", tail=12)

$ -m pytest -q
.......................................                                  [100%]
39 passed in 0.70s


0

---
## Паук 1: новости (статический сайт)

Страницы отдаются сервером готовыми, браузер не нужен. Пагинация — по номеру страницы;
паук добирает страницы, пока не наберёт 100 записей, прошедших фильтр и дедупликацию.

In [16]:
print((LAB / "webmining/spiders/news.py").read_text(encoding="utf-8"))

import scrapy
from itemloaders.processors import Join, MapCompose, TakeFirst
from scrapy import signals
from scrapy.loader import ItemLoader

from webmining.items import NewsItem
from webmining.sites import NEWS_SITES

MAX_PAGES = 30

class NewsLoader(ItemLoader):
    default_item_class = NewsItem
    default_input_processor = MapCompose(str.strip)
    default_output_processor = TakeFirst()
    summary_out = Join(" ")

class NewsSpider(scrapy.Spider):
    name = "news"

    def __init__(self, site="habr", keywords="", limit=100, *args, **kwargs):
        super().__init__(*args, **kwargs)
        if site not in NEWS_SITES:
            raise ValueError(f"неизвестный сайт {site}, доступны: {', '.join(NEWS_SITES)}")

        self.site = site
        self.profile = NEWS_SITES[site]
        self.keywords = [word.strip().lower() for word in keywords.split(",") if word.strip()]
        self.limit = int(limit)
        self.skipped = 0
        self.collected = 0
        self.page = 1

    # item

In [17]:
run("run_spider.py", "--spider", "news", "--keywords", "ИИ,нейросети,данные",
    "--limit", LIMIT, "--output", "news.json", tail=6)

$ run_spider.py --spider news --keywords ИИ,нейросети,данные --limit 100 --output news.json
Паук news запущен, лог: c:\projects\web-mining\lab4\logs\news.log
news.csv: 184433 байт
news.json: 192856 байт


0

Отчёты pipelines из лога — сколько записей поправлено и отброшено:

In [18]:
pipeline_report("news")

    INFO: ValidationPipeline: отброшено записей — 0
    INFO: DuplicationPipeline: отброшено дубликатов — 0
    INFO: CleaningPipeline: поправлено полей — 105
    INFO: Не подошло по ключевым словам — 49 записей
    INFO: ValidationPipeline: отброшено записей — 0
    INFO: DuplicationPipeline: отброшено дубликатов — 0
    INFO: CleaningPipeline: поправлено полей — 152
    INFO: Не подошло по ключевым словам — 60 записей
    INFO: ValidationPipeline: отброшено записей — 0
    INFO: DuplicationPipeline: отброшено дубликатов — 0
    INFO: CleaningPipeline: поправлено полей — 152
    INFO: Не подошло по ключевым словам — 60 записей


Что собралось:

In [19]:
news = json.loads((LAB / "output/news.json").read_text(encoding="utf-8"))
print("записей:", len(news), "| уникальных адресов:", len({row["url"] for row in news}))
print()
for row in news[:5]:
    print(f'{row["date"][:10]}  {row["title"][:70]}')
    print(f'    {row["url"]}')

записей: 120 | уникальных адресов: 120

2026-10-02  Бусти.ру — или как зарубежная компания пытается оставаться российской
    https://habr.com/ru/articles/1089796/
2026-10-02  Крошечная нейронная сеть без компьютера и калькулятора
    https://habr.com/ru/articles/1086410/
2026-10-02  Как я создал СуперЗавуч — программу для генерации расписаний занятий
    https://habr.com/ru/articles/1089006/
2026-10-02  Разбираемся с MCP на практике: сервер документации Swagger на Laravel
    https://habr.com/ru/articles/1089476/
2026-10-02  Typing в Python 3.11–3.14: что изменилось и как это использовать уже с
    https://habr.com/ru/articles/1089780/


CSV с теми же данными — для Excel и аналитики:

In [20]:
csv_text = (LAB / "output/news.csv").read_text(encoding="utf-8")
print("\n".join(csv_text.splitlines()[:4]))

title,url,date,summary,source
Бусти.ру — или как зарубежная компания пытается оставаться российской,https://habr.com/ru/articles/1089796/,2026-10-02T16:47:23.000Z,"есть такой удивительный интернет-феномен: сервис может быть придуман в россии, создан российской компанией, зарабатывать на российской аудитории, принимать деньги российских пользователей, учитывать российские законы — и при этом юридически иметь к россии всё меньше и меньше отношения. сегодня речь про boosty. точнее, про boosty.to — именно так называется домен сервиса, который многие по привычке ищут как «бусти.ру». я решил немного покопаться в его биографии. и чем глубже копал, тем интереснее становилась конструкция. потому что история boosty - это уже не просто история российского аналога patreon. это путешествие длиной в несколько юрисдикций:",habr
Крошечная нейронная сеть без компьютера и калькулятора,https://habr.com/ru/articles/1086410/,2026-10-02T16:42:12.000Z,"""- когда я вижу ответ llm, я скорее поверю, что где-то 

---
## Паук 2: товары (динамический сайт)

Магазин отдаёт пустой контейнер, карточки дорисовывает JavaScript. Запрос помечен
`meta={"playwright": True}`, браузер ждёт первую карточку и прокручивает страницу,
чтобы подгрузились остальные товары.

In [21]:
print((LAB / "webmining/spiders/ecommerce.py").read_text(encoding="utf-8"))

import re

import scrapy
from itemloaders.processors import Identity, MapCompose, TakeFirst
from scrapy.loader import ItemLoader
from scrapy_playwright.page import PageMethod

from webmining.items import ProductItem
from webmining.sites import SHOP_SITES

SCROLL_STEPS = 12
SCROLL_PAUSE = 1000
PRICE = re.compile(r"\d[\d\s .,]*\d|\d")

class ProductLoader(ItemLoader):
    default_item_class = ProductItem
    default_input_processor = MapCompose(str.strip)
    default_output_processor = TakeFirst()
    # числа уже разобраны, strip к ним применять нельзя
    price_in = Identity()
    rating_in = Identity()
    reviews_in = Identity()

# «$1,139.54», «1 234,56 ₽», «от 89 990 ₽» — из всего этого нужна одна цифра
def parse_price(text):
    if not text:
        return None

    match = PRICE.search(text.replace(" ", " "))
    if not match:
        return None

    number = match.group().replace(" ", "")
    if "," in number and "." in number:
        number = number.replace(",", "")      # 1,1

In [22]:
run("run_spider.py", "--spider", "ecommerce", "--category", "laptops",
    "--limit", LIMIT, "--output", "prices.json", tail=6)

$ run_spider.py --spider ecommerce --category laptops --limit 100 --output prices.json
Паук ecommerce запущен, лог: c:\projects\web-mining\lab4\logs\ecommerce.log
prices.csv: 4313 байт
prices.json: 7036 байт


0

In [23]:
pipeline_report("ecommerce")

    INFO: ValidationPipeline: отброшено записей — 0
    INFO: DuplicationPipeline: отброшено дубликатов — 0
    INFO: CleaningPipeline: поправлено полей — 0
    INFO: Карточек на странице: 39 (https://webscraper.io/test-sites/e-commerce/scroll/computers/laptops)
    INFO: ValidationPipeline: отброшено записей — 0
    INFO: DuplicationPipeline: отброшено дубликатов — 0
    INFO: CleaningPipeline: поправлено полей — 0
    INFO: Карточек на странице: 39 (https://webscraper.io/test-sites/e-commerce/scroll/computers/laptops)
    INFO: ValidationPipeline: отброшено записей — 0
    INFO: DuplicationPipeline: отброшено дубликатов — 0
    INFO: CleaningPipeline: поправлено полей — 0


Собранные товары и разброс цен:

In [24]:
products = json.loads((LAB / "output/prices.json").read_text(encoding="utf-8"))
prices = [row["price"] for row in products if row.get("price")]

print("товаров:", len(products), "| с ценой:", len(prices))
if prices:
    print(f"цена: от {min(prices):.2f} до {max(prices):.2f}, "
          f"в среднем {sum(prices) / len(prices):.2f} {products[0]['currency']}")
print()
for row in sorted(products, key=lambda item: -(item.get("price") or 0))[:5]:
    print(f'{row["price"]:>10.2f}  рейтинг {row.get("rating")}  {row["title"][:50]}')

товаров: 39 | с ценой: 39
цена: от 295.99 до 520.99, в среднем 420.11 USD

    520.99  рейтинг None  HP 250 G3
    498.23  рейтинг None  Lenovo V510 Black
    497.17  рейтинг None  Dell Vostro 15 (3568) Red
    494.71  рейтинг None  Acer Aspire 3 A315-51 Black
    488.78  рейтинг None  Dell Vostro 15


---
## Проверка качества данных

Ровно то, ради чего нужны pipelines: ни пустых обязательных полей, ни дубликатов,
ни битых кодировок.

In [25]:
def check(records, name):
    urls = [row["url"] for row in records]
    empty = [row for row in records if not row.get("title") or not row.get("url")]
    mojibake = [row for row in records if "Ð" in row.get("title", "") or "â€" in row.get("title", "")]

    print(f"{name}: записей {len(records)}")
    print(f"   дубликатов адресов: {len(urls) - len(set(urls))}")
    print(f"   записей с пустыми обязательными полями: {len(empty)}")
    print(f"   заголовков с битой кодировкой: {len(mojibake)}")
    print(f"   набрано минимум 100: {'да' if len(records) >= 100 else 'нет'}")

check(news, "Новости")
check(products, "Товары")

Новости: записей 120
   дубликатов адресов: 0
   записей с пустыми обязательными полями: 0
   заголовков с битой кодировкой: 0
   набрано минимум 100: да
Товары: записей 39
   дубликатов адресов: 0
   записей с пустыми обязательными полями: 0
   заголовков с битой кодировкой: 0
   набрано минимум 100: нет
